In [ ]:
# load common model definitions

%run 04_models_ABC.ipynb


In [ ]:
# paths and settings

CURTAILMENT_RESULTS_DIR = (
    Path("results") / "curtailment_sensitivity"
)

CURTAILMENT_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CURTAILMENT_TIMESTAMPS_FILE = (
    PROCESSED_DIR
    / "penmanshiel_excluded_curtailment_timestamps.parquet"
)

SEEDS = [1, 21, 42, 84, 123]

SENSITIVITY_MAX_EPOCHS = 500
SENSITIVITY_PATIENCE = 30

SENSITIVITY_LEARNING_RATE_A = 1e-3
SENSITIVITY_LEARNING_RATE_B = 1e-3
SENSITIVITY_LEARNING_RATE_C = 1e-3

SENSITIVITY_HIDDEN_A = (64, 32)
SENSITIVITY_HIDDEN_B = (64, 32)
SENSITIVITY_HIDDEN_C = (64, 32)


In [ ]:
# load the modelling data and curtailment timestamps

scada = pd.read_parquet(
    SCADA_FILE
).copy()

static = pd.read_parquet(
    STATIC_FILE
).copy()

farm_data, turbine_ids = build_modelling_data(
    scada
)

train_full, validation_data, _ = chronological_split(
    farm_data
)

excluded_curtailment_df = pd.read_parquet(
    CURTAILMENT_TIMESTAMPS_FILE
)

excluded_curtailment_timestamps = pd.DatetimeIndex(
    pd.to_datetime(
        excluded_curtailment_df["timestamp"],
        utc=True,
    )
).drop_duplicates()

train_excluded = train_full.loc[
    ~train_full.index.isin(
        excluded_curtailment_timestamps
    )
].copy()

print(f"training rows retained: {len(train_full):,}")
print(f"training rows excluded: {len(train_excluded):,}")
print(f"validation rows: {len(validation_data):,}")


In [ ]:
# train one sensitivity model with early stopping

def train_sensitivity_model(
    X_train,
    y_train,
    X_validation,
    y_validation,
    hidden_layers,
    learning_rate,
    seed,
):
    set_seed(seed)

    X_train_tensor = torch.tensor(
        X_train,
        dtype=torch.float32,
    )

    y_train_tensor = torch.tensor(
        y_train,
        dtype=torch.float32,
    )

    X_validation_tensor = torch.tensor(
        X_validation,
        dtype=torch.float32,
    )

    y_validation_tensor = torch.tensor(
        y_validation,
        dtype=torch.float32,
    )

    model = build_network(
        input_size=X_train_tensor.shape[1],
        output_size=y_train_tensor.shape[1],
        hidden_layers=hidden_layers,
    )

    loss_function = nn.HuberLoss(
        delta=1.0
    )

    optimiser = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
    )

    generator = torch.Generator()
    generator.manual_seed(seed)

    loader = DataLoader(
        TensorDataset(
            X_train_tensor,
            y_train_tensor,
        ),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
    )

    best_validation_loss = np.inf
    best_state = None
    best_epoch = None
    epochs_without_improvement = 0

    for epoch in range(
        SENSITIVITY_MAX_EPOCHS
    ):
        model.train()

        for X_batch, y_batch in loader:
            optimiser.zero_grad()

            prediction = model(X_batch)

            loss = loss_function(
                prediction,
                y_batch,
            )

            loss.backward()
            optimiser.step()

        model.eval()

        with torch.no_grad():
            validation_loss = loss_function(
                model(X_validation_tensor),
                y_validation_tensor,
            ).item()

        if validation_loss < best_validation_loss:
            best_validation_loss = validation_loss
            best_epoch = epoch + 1
            best_state = copy.deepcopy(
                model.state_dict()
            )
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if (
            epochs_without_improvement
            >= SENSITIVITY_PATIENCE
        ):
            break

    model.load_state_dict(best_state)
    model.eval()

    return model, best_epoch


In [ ]:
# evaluate one retained or excluded training condition

def run_curtailment_condition(
    training_data,
    condition,
    seed,
):
    feature_columns = [
        "global_ws",
        "wd_sin",
        "wd_cos",
    ]

    wind_speed_columns = [
        f"ws_t{turbine_id}"
        for turbine_id in turbine_ids
    ]

    turbine_power_columns = [
        f"power_t{turbine_id}"
        for turbine_id in turbine_ids
    ]

    X_train_raw = training_data[
        feature_columns
    ].to_numpy()

    X_validation_raw = validation_data[
        feature_columns
    ].to_numpy()

    X_mean, X_std = fit_standardisation(
        X_train_raw
    )

    X_train = standardise(
        X_train_raw,
        X_mean,
        X_std,
    )

    X_validation = standardise(
        X_validation_raw,
        X_mean,
        X_std,
    )

    farm_capacity_kw = static.loc[
        static["turbine_id"].isin(
            turbine_ids
        ),
        "rated_power_kw",
    ].sum()

    # model A

    y_A_train_raw = training_data[
        ["total_power"]
    ].to_numpy()

    y_A_validation_raw = validation_data[
        ["total_power"]
    ].to_numpy()

    y_A_mean, y_A_std = fit_standardisation(
        y_A_train_raw
    )

    model_A, epoch_A = train_sensitivity_model(
        X_train=X_train,
        y_train=standardise(
            y_A_train_raw,
            y_A_mean,
            y_A_std,
        ),
        X_validation=X_validation,
        y_validation=standardise(
            y_A_validation_raw,
            y_A_mean,
            y_A_std,
        ),
        hidden_layers=SENSITIVITY_HIDDEN_A,
        learning_rate=SENSITIVITY_LEARNING_RATE_A,
        seed=seed,
    )

    with torch.no_grad():
        pred_A_validation = (
            model_A(
                torch.tensor(
                    X_validation,
                    dtype=torch.float32,
                )
            ).numpy()
            * y_A_std
            + y_A_mean
        ).reshape(-1)

    pred_A_validation = np.clip(
        pred_A_validation,
        0,
        farm_capacity_kw,
    )

    # model B

    y_B_train_raw = training_data[
        wind_speed_columns
    ].to_numpy()

    y_B_validation_raw = validation_data[
        wind_speed_columns
    ].to_numpy()

    y_B_mean, y_B_std = fit_standardisation(
        y_B_train_raw
    )

    model_B, epoch_B = train_sensitivity_model(
        X_train=X_train,
        y_train=standardise(
            y_B_train_raw,
            y_B_mean,
            y_B_std,
        ),
        X_validation=X_validation,
        y_validation=standardise(
            y_B_validation_raw,
            y_B_mean,
            y_B_std,
        ),
        hidden_layers=SENSITIVITY_HIDDEN_B,
        learning_rate=SENSITIVITY_LEARNING_RATE_B,
        seed=seed,
    )

    with torch.no_grad():
        pred_B_validation_ws = (
            model_B(
                torch.tensor(
                    X_validation,
                    dtype=torch.float32,
                )
            ).numpy()
            * y_B_std
            + y_B_mean
        )

    power_curves = build_power_curves(
        scada=scada,
        training_timestamps=training_data.index,
        turbine_ids=turbine_ids,
    )

    pred_B_validation_turbine_power = (
        speeds_to_turbine_power(
            pred_B_validation_ws,
            power_curves,
            turbine_ids,
        )
    )

    pred_B_validation_farm = np.clip(
        pred_B_validation_turbine_power.sum(axis=1),
        0,
        farm_capacity_kw,
    )

    # model C

    y_C_train_raw = training_data[
        turbine_power_columns
    ].to_numpy()

    y_C_validation_raw = validation_data[
        turbine_power_columns
    ].to_numpy()

    y_C_mean, y_C_std = fit_standardisation(
        y_C_train_raw
    )

    model_C, epoch_C = train_sensitivity_model(
        X_train=X_train,
        y_train=standardise(
            y_C_train_raw,
            y_C_mean,
            y_C_std,
        ),
        X_validation=X_validation,
        y_validation=standardise(
            y_C_validation_raw,
            y_C_mean,
            y_C_std,
        ),
        hidden_layers=SENSITIVITY_HIDDEN_C,
        learning_rate=SENSITIVITY_LEARNING_RATE_C,
        seed=seed,
    )

    with torch.no_grad():
        pred_C_validation_turbines = (
            model_C(
                torch.tensor(
                    X_validation,
                    dtype=torch.float32,
                )
            ).numpy()
            * y_C_std
            + y_C_mean
        )

    pred_C_validation_farm = np.clip(
        pred_C_validation_turbines.sum(axis=1),
        0,
        farm_capacity_kw,
    )

    actual_validation_farm = validation_data[
        "total_power"
    ].to_numpy()

    return {
        "condition": condition,
        "seed": seed,
        "n_train": len(training_data),
        "A_MAE_kW": mean_absolute_error(
            actual_validation_farm,
            pred_A_validation,
        ),
        "A_RMSE_kW": np.sqrt(
            mean_squared_error(
                actual_validation_farm,
                pred_A_validation,
            )
        ),
        "B_MAE_kW": mean_absolute_error(
            actual_validation_farm,
            pred_B_validation_farm,
        ),
        "B_RMSE_kW": np.sqrt(
            mean_squared_error(
                actual_validation_farm,
                pred_B_validation_farm,
            )
        ),
        "B_ws_MAE_ms": mean_absolute_error(
            y_B_validation_raw.reshape(-1),
            pred_B_validation_ws.reshape(-1),
        ),
        "B_ws_R2": r2_score(
            y_B_validation_raw.reshape(-1),
            pred_B_validation_ws.reshape(-1),
        ),
        "C_MAE_kW": mean_absolute_error(
            actual_validation_farm,
            pred_C_validation_farm,
        ),
        "C_RMSE_kW": np.sqrt(
            mean_squared_error(
                actual_validation_farm,
                pred_C_validation_farm,
            )
        ),
        "A_best_epoch": epoch_A,
        "B_best_epoch": epoch_B,
        "C_best_epoch": epoch_C,
    }


In [ ]:
# run retained and excluded conditions across five seeds

sensitivity_rows = []

for seed in SEEDS:
    sensitivity_rows.append(
        run_curtailment_condition(
            training_data=train_full,
            condition="Retained",
            seed=seed,
        )
    )

    sensitivity_rows.append(
        run_curtailment_condition(
            training_data=train_excluded,
            condition="Excluded",
            seed=seed,
        )
    )

seed_results = pd.DataFrame(
    sensitivity_rows
)

seed_results.to_csv(
    CURTAILMENT_RESULTS_DIR
    / "curtailment_sensitivity_seed_results.csv",
    index=False,
)

print(
    seed_results
    .round(3)
    .to_string(index=False)
)


In [ ]:
# summarise the validation sensitivity

summary_rows = []

for condition in [
    "Retained",
    "Excluded",
]:
    condition_data = seed_results[
        seed_results["condition"] == condition
    ]

    summary_rows.append(
        {
            "condition": condition,
            "A_MAE_mean_kW": condition_data["A_MAE_kW"].mean(),
            "A_MAE_sd_kW": condition_data["A_MAE_kW"].std(ddof=1),
            "B_MAE_mean_kW": condition_data["B_MAE_kW"].mean(),
            "B_MAE_sd_kW": condition_data["B_MAE_kW"].std(ddof=1),
            "B_ws_MAE_mean_ms": condition_data["B_ws_MAE_ms"].mean(),
            "B_ws_MAE_sd_ms": condition_data["B_ws_MAE_ms"].std(ddof=1),
            "C_MAE_mean_kW": condition_data["C_MAE_kW"].mean(),
            "C_MAE_sd_kW": condition_data["C_MAE_kW"].std(ddof=1),
        }
    )

summary = pd.DataFrame(
    summary_rows
)

summary.to_csv(
    CURTAILMENT_RESULTS_DIR
    / "curtailment_sensitivity_summary.csv",
    index=False,
)

comparison_rows = []

for model_name in [
    "A",
    "B",
    "C",
]:
    retained = summary.loc[
        summary["condition"] == "Retained",
        f"{model_name}_MAE_mean_kW",
    ].iloc[0]

    excluded = summary.loc[
        summary["condition"] == "Excluded",
        f"{model_name}_MAE_mean_kW",
    ].iloc[0]

    change = excluded - retained

    comparison_rows.append(
        {
            "model": f"Model {model_name}",
            "retained_MAE_kW": retained,
            "excluded_MAE_kW": excluded,
            "change_kW": change,
            "change_pct": (
                change / retained * 100
            ),
        }
    )

comparison = pd.DataFrame(
    comparison_rows
)

comparison.to_csv(
    CURTAILMENT_RESULTS_DIR
    / "curtailment_sensitivity_comparison.csv",
    index=False,
)

print(summary.round(3).to_string(index=False))
print()
print(comparison.round(2).to_string(index=False))
